<h1>LD Pipeline 2024 (Version 1.0, PROD)</h1>

<h3>Lade die Umgebung</h3>

In [1]:
import os
import requests

try:
    initialized
except NameError:
    initialized = False

if not initialized:
    initialized = True
    os.chdir('../')
    import main
    from pipeline.base import Utils
    Utils.is_jupyter_notebook = True

<h3>Generiere die Triple-Dateien</h3>

In [4]:
names = [ 'code', 'cube', 'groupCode', 'hierarchy', 'legalFoundation',
         'measureUnit', 'measure', 'property', 'room', 'time', 'observation' ]

names = ['time']

options = {
    'db_batch_size': 100000,
    'write_batch_size': 600000,
    'max_iteration': None
}

options = {
    'db_batch_size': 1,
    'write_batch_size': 1,
    'max_iteration': 1
}

for name in names:
    main.step(name=f"{name}Templating", env=main.Env.prod, options=options)

2024-06-13 08:42:56 - Creating temporary table #view_time ...
2024-06-13 08:42:57 - Creating an index on the _sort_order column ...
2024-06-13 08:42:58 - done
2024-06-13 08:42:58 - Downloading data ...
2024-06-13 08:42:58 - 1 rows downloaded in the 1. iteration.
2024-06-13 08:42:58 - Generating triples ...
2024-06-13 08:42:59 - done
2024-06-13 08:42:59 - Writing batch data to view_time_20240613084258_975b340d-6679-4c47-b232-7092eead61c7.nt.gz ...
2024-06-13 08:42:59 - File is created.
2024-06-13 08:42:59 - 1. iteration is finished.


<h3>Setze den Fuseki-Server zurück (Optional)</h3>

In [ ]:
def reset_fuseki_server():
    server_url = 'http://localhost:8080/fuseki/ssz/update'
    query = """
        DELETE WHERE {
            ?s ?p ?o.
        }
    """
    response = requests.post(server_url, data={'update': query})
    if response.status_code == 200:
        print("All triples are successfully deleted.")
    else:
        print(f"Status code: {response.status_code}, Error: {response.text}")

reset_fuseki_server()

<h3>Übertrage die Dateien in der Inbox zu dem lokalen Fuseki-Server</h3>

In [3]:
main.step(name="uploadToFuseki", env=main.Env.prod)

2024-06-13 08:42:42 - Uploading view_time_20240613084204_ff6169fd-cc9d-4d8e-9dc8-164994c5622f.nt.gz
2024-06-13 08:42:42 - OK


<h3>Übertrage die Dateien in der Inbox zu Stardog</h3>

In [ ]:
main.step(name="uploadToStardog", env=main.Env.prod)

<h3>SPARQL-Abfragen an Stardog senden</h3>

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

stardog_graph_uri = utils.get_stardog_graph_uri(env=main.Env.prod)
df = utils.execute_sparql(f"""
    SELECT * FROM <{stardog_graph_uri}> WHERE {{
        <https://ld.integ.stadt-zuerich.ch/statistics/observation/BAP1045-GBA1200-XXX0000-XXX0000-XXX0000-R00073-Z31122012> ?pred ?obj
    }} LIMIT 100
""", env=main.Env.prod)
display(df)